# Palaeogeography

## Loading packages

In [1]:
library("palaeoverse")
library("divvy")
library("lwgeom")
library("h3jsr")
library("ggplot2")
library("ggpubr")
library("sf")
library("raster")
library("geojsonio")
library("sp")
library("dplyr") 

Linking to liblwgeom 3.0.0beta1 r16016, GEOS 3.8.0, PROJ 6.3.1

Linking to GEOS 3.8.0, GDAL 3.0.4, PROJ 6.3.1; sf_use_s2() is TRUE


Attachement du package : ‘sf’


L'objet suivant est masqué depuis ‘package:lwgeom’:

    st_perimeter


Le chargement a nécessité le package : sp

Registered S3 method overwritten by 'geojson':
  method        from     
  print.geojson geojsonsf


Attachement du package : ‘geojsonio’


L'objet suivant est masqué depuis ‘package:base’:

    pretty



Attachement du package : ‘dplyr’


Les objets suivants sont masqués depuis ‘package:raster’:

    intersect, select, union


Les objets suivants sont masqués depuis ‘package:stats’:

    filter, lag


Les objets suivants sont masqués depuis ‘package:base’:

    intersect, setdiff, setequal, union




## Setting up environement

In [2]:
sf_use_s2(FALSE)

Spherical geometry (s2) switched off



## Loading occurence data

### Extinct taxa

In [3]:
table_coordinate_extinct <- read.table("Occurences_extinct_Orectolobiformes.tsv", header = TRUE, sep ="\t")

In [4]:
table_coordinate_extinct <- cbind(table_coordinate_extinct[, c(4)], apply(table_coordinate_extinct[,c(6,7)], 1, mean), table_coordinate_extinct[, c(11, 12)])

In [5]:
colnames(table_coordinate_extinct) <- c("scientificName", "age", colnames(table_coordinate_extinct)[c(3,4)])

In [6]:
table_coordinate_extinct

scientificName,age,Latitude,Longitude
<chr>,<dbl>,<dbl>,<dbl>
Chiloscyllium sp.,56.3205,34.713995,7.959418
Ginglymostoma khouribgaense,56.3205,34.713995,7.959418
Ginglymostoma sp.,56.3205,34.713995,7.959418
Nebrius obliquus,56.3205,34.713995,7.959418
Palaeorhincodon dartevellei,56.3205,34.713995,7.959418
Delpitoscyllium africanum,58.8105,-5.140000,12.080000
Delpitoscyllium africanum,60.4005,-5.140000,12.080000
Ginglymostoma angolense,44.4005,-5.056395,12.321175
Ginglymostoma angolense,44.4005,-5.140000,12.080000


### Extant taxa

In [7]:
table_coordinate_extant <- read.table("Occurences_extant_Orectolobiformes.tsv", header = TRUE, sep ="\t")

## Paleocoordinate

In [8]:
paleo_coordinate <- palaeorotate(table_coordinate_extinct, lng = "Longitude", lat = "Latitude", age = "age", model = c("MERDITH2021", "PALEOMAP", "MATTHEWS2016_pmag_ref"))

MERDITH2021

PALEOMAP

MATTHEWS2016_pmag_ref

Warning message in palaeorotate(table_coordinate_extinct, lng = "Longitude", lat = "Latitude", :
“Palaeocoordinates could not be reconstructed for all points.
Either assigned plate does not exist at time of reconstruction or the Global Plate Model(s) does not cover the age of reconstruction.”


In [9]:
paleo_coordinate_cleaned <- na.omit(paleo_coordinate)

In [10]:
data_epochs <- data.frame(Epoch = c("Today_Miocene", "Oligocene", "Eocene", "Palaeocene", "Campanian_Maastrichian", "Cenomanian_Santonian", "Early Cretaceous", "Jurassic"),
        Age_min = c(0, 23.031, 33.901, 56.001, 66.001, 86.3, 100.501, 145.001), 
        Age_max = c(23.03, 33.9, 56, 66, 86.3, 100.5, 145, 201.4), 
        Age = c(mean(0, 23.03), mean(23.03, 33.9), mean(33.9, 56), mean(56, 66), mean(66, 86.3), mean(86.3, 100.5), mean(100.5, 145), mean(145,201.4)))

In [11]:
# Set up bounding box
ras <- raster::raster(res = 5, val = 1)
ras <- rasterToPolygons(x = ras, dissolve = TRUE)

# Robinson projection
bb <- sf::st_as_sf(x = ras)
bb <- st_transform(x = bb, crs = sf::st_crs(4326))
bbox <- st_graticule(crs = st_crs("ESRI:54030"), lat = c(-89.9, 89.9), lon = c(-179.9, 179.9))

In [12]:
for(i in 1:nrow(data_epochs)){
    temp_url <- paste("https://gws.gplates.org/reconstruct/coastlines/?&time=", data_epochs$Age[i], "&model=MERDITH2021&anchor_plate_id=1", sep = "")
    temp_name <- paste("Plot/Merdith_2021/paleocoordinates", data_epochs$Epoch[i], ".pdf", sep ="")
    gpm <- read_sf(temp_url)
    gpm <- gpm  %>% st_union()
    
    temp_paleo_data <- paleo_coordinate_cleaned[paleo_coordinate_cleaned$age > data_epochs$Age_min[i] & paleo_coordinate_cleaned$age < data_epochs$Age_max[i], ]
    pal_cor <- data.frame(longitude = temp_paleo_data$p_lng_MERDITH2021, latitude = temp_paleo_data$p_lat_MERDITH2021)
    
    coordinates(pal_cor) <- ~longitude + latitude
    proj4string(pal_cor) <- CRS("+proj=longlat +datum=WGS84")
    SpatialPoints(pal_cor, proj4string = CRS("ESRI:54030"))
    coords_robinson <- spTransform(pal_cor, CRS("+proj=robin +datum=WGS84"))
    coords_sf <- st_as_sf(coords_robinson)
    
    palaeo_coordinates <- gpm  %>% ggplot() +
        geom_sf(data = bb, fill = "#ADD8E6", colour = NA) +
        geom_sf(data = gpm, fill = "#BEB4B0", colour = "black", alpha = 1) +
        geom_sf(data = bbox) +
        geom_sf(data = coords_sf, aes(color = "#FAC898")) +
        coord_sf(crs = sf::st_crs("ESRI:54030")) +
        theme_void()
    
    ggsave(filename = temp_name, plot = palaeo_coordinates)
}

although coordinates are longitude/latitude, st_union assumes that they are
planar

Saving 6.67 x 6.67 in image
although coordinates are longitude/latitude, st_union assumes that they are
planar

Saving 6.67 x 6.67 in image
although coordinates are longitude/latitude, st_union assumes that they are
planar

Saving 6.67 x 6.67 in image
although coordinates are longitude/latitude, st_union assumes that they are
planar

Saving 6.67 x 6.67 in image
although coordinates are longitude/latitude, st_union assumes that they are
planar

Saving 6.67 x 6.67 in image
although coordinates are longitude/latitude, st_union assumes that they are
planar

Saving 6.67 x 6.67 in image
although coordinates are longitude/latitude, st_union assumes that they are
planar

Saving 6.67 x 6.67 in image
although coordinates are longitude/latitude, st_union assumes that they are
planar

Saving 6.67 x 6.67 in image


In [13]:
for(i in 1:nrow(data_epochs)){
    temp_url <- paste("https://gws.gplates.org/reconstruct/coastlines/?&time=", data_epochs$Age[i], "&model=PALEOMAP&anchor_plate_id=1", sep = "")
    temp_name <- paste("Plot/PALEOMAP/paleocoordinates", data_epochs$Epoch[i], ".pdf", sep ="")
    gpm <- read_sf(temp_url)
    gpm <- gpm  %>% st_union()
    
    temp_paleo_data <- paleo_coordinate_cleaned[paleo_coordinate_cleaned$age > data_epochs$Age_min[i] & paleo_coordinate_cleaned$age < data_epochs$Age_max[i], ]
    pal_cor <- data.frame(longitude = temp_paleo_data$p_lng_PALEOMAP, latitude = temp_paleo_data$p_lat_PALEOMAP)
    
    coordinates(pal_cor) <- ~longitude + latitude
    proj4string(pal_cor) <- CRS("+proj=longlat +datum=WGS84")
    SpatialPoints(pal_cor, proj4string = CRS("ESRI:54030"))
    coords_robinson <- spTransform(pal_cor, CRS("+proj=robin +datum=WGS84"))
    coords_sf <- st_as_sf(coords_robinson)
    
    palaeo_coordinate <- gpm  %>% ggplot() +
        geom_sf(data = bb, fill = "#ADD8E6", colour = NA) +
        geom_sf(data = gpm, fill = "#BEB4B0", colour = "black", alpha = 1) +
        geom_sf(data = bbox) +
        geom_sf(data = coords_sf, aes(color = "#FAC898")) +
        coord_sf(crs = sf::st_crs("ESRI:54030")) +
        theme_void()
    
    ggsave(filename = temp_name, plot = palaeo_coordinate)
}

although coordinates are longitude/latitude, st_union assumes that they are
planar

Saving 6.67 x 6.67 in image
although coordinates are longitude/latitude, st_union assumes that they are
planar

Saving 6.67 x 6.67 in image
although coordinates are longitude/latitude, st_union assumes that they are
planar

Saving 6.67 x 6.67 in image
although coordinates are longitude/latitude, st_union assumes that they are
planar

Saving 6.67 x 6.67 in image
although coordinates are longitude/latitude, st_union assumes that they are
planar

Saving 6.67 x 6.67 in image
although coordinates are longitude/latitude, st_union assumes that they are
planar

Saving 6.67 x 6.67 in image
although coordinates are longitude/latitude, st_union assumes that they are
planar

Saving 6.67 x 6.67 in image
although coordinates are longitude/latitude, st_union assumes that they are
planar

Saving 6.67 x 6.67 in image


In [14]:
for(i in 1:nrow(data_epochs)){
    temp_url <- paste("https://gws.gplates.org/reconstruct/coastlines/?&time=", data_epochs$Age[i], "&model=PALEOMAP&anchor_plate_id=1", sep = "")
    temp_name <- paste("Plot/PALEOMAP/paleocoordinates", data_epochs$Epoch[i], ".pdf", sep ="")
    gpm <- read_sf(temp_url)
    gpm <- gpm  %>% st_union()
    
    temp_paleo_data <- paleo_coordinate_cleaned[paleo_coordinate_cleaned$age > data_epochs$Age_min[i] & paleo_coordinate_cleaned$age < data_epochs$Age_max[i], ]
    pal_cor <- data.frame(longitude = temp_paleo_data$p_lng_PALEOMAP, latitude = temp_paleo_data$p_lat_PALEOMAP)
    
    coordinates(pal_cor) <- ~longitude + latitude
    proj4string(pal_cor) <- CRS("+proj=longlat +datum=WGS84")
    SpatialPoints(pal_cor, proj4string = CRS("ESRI:54030"))
    coords_robinson <- spTransform(pal_cor, CRS("+proj=robin +datum=WGS84"))
    coords_sf <- st_as_sf(coords_robinson)
    
    palaeo_coordinate <- gpm  %>% ggplot() +
        geom_sf(data = bb, fill = "#ADD8E6", colour = NA) +
        geom_sf(data = gpm, fill = "#BEB4B0", colour = "black", alpha = 1) +
        geom_sf(data = bbox) +
        geom_sf(data = coords_sf, aes(color = "#FAC898")) +
        coord_sf(crs = sf::st_crs("ESRI:54030")) +
        theme_void()
    
    ggsave(filename = temp_name, plot = palaeo_coordinate)
}

although coordinates are longitude/latitude, st_union assumes that they are
planar

Saving 6.67 x 6.67 in image
although coordinates are longitude/latitude, st_union assumes that they are
planar

Saving 6.67 x 6.67 in image
although coordinates are longitude/latitude, st_union assumes that they are
planar

Saving 6.67 x 6.67 in image
although coordinates are longitude/latitude, st_union assumes that they are
planar

Saving 6.67 x 6.67 in image
although coordinates are longitude/latitude, st_union assumes that they are
planar

Saving 6.67 x 6.67 in image
although coordinates are longitude/latitude, st_union assumes that they are
planar

Saving 6.67 x 6.67 in image
although coordinates are longitude/latitude, st_union assumes that they are
planar

Saving 6.67 x 6.67 in image
although coordinates are longitude/latitude, st_union assumes that they are
planar

Saving 6.67 x 6.67 in image


In [15]:
data_epochs <- data.frame(Epoch = c(seq(0, 265, by = 5)),
        Age = c(seq(0, 265, by = 5)))

In [16]:
data_epochs<- data_epochs[!data_epochs$Age==55,]
data_epochs<- data_epochs[!data_epochs$Age==75,]
data_epochs<- data_epochs[!data_epochs$Age==100,]
data_epochs<- data_epochs[!data_epochs$Age==110,]
data_epochs<- data_epochs[!data_epochs$Age==115,]
data_epochs<- data_epochs[!data_epochs$Age==145,]
data_epochs<- data_epochs[!data_epochs$Age==175,]
data_epochs<- data_epochs[!data_epochs$Age==190,]
data_epochs<- data_epochs[!data_epochs$Age==210,]
data_epochs<- data_epochs[!data_epochs$Age==215,]
data_epochs<- data_epochs[!data_epochs$Age==225,]
data_epochs<- data_epochs[!data_epochs$Age==235,]

In [18]:
for(i in 1:nrow(data_epochs)){
    temp_CM <- paste("PaleoDEMS/CM/", data_epochs$Age[i], "Ma_CM_v7.shp", sep = "")
    temp_name <- paste("Plot/PALEOMAP_DEMS/paleocoordinates", data_epochs$Epoch[i], ".pdf", sep ="")
    temp_CS <- paste("PaleoDEMS/CS/", data_epochs$Age[i], "Ma_CS_v7.shp", sep = "")

    gpm <- read_sf(temp_CM)

    gpm2 <- read_sf(temp_CS)
    
    palaeo_coordinate <- gpm  %>% ggplot() +         
        geom_sf(data = bb, fill = "#ADD8E6", colour = NA) +
        geom_sf(data = gpm, fill = "#81CDC6", colour = "black", alpha = 1) +
        geom_sf(data = gpm2, fill = "#BEB4B0", colour = "black", alpha = 1) +
        geom_sf(data = bbox) +
        coord_sf(crs = sf::st_crs("ESRI:54030")) +
        theme_void()
    
    ggsave(filename = temp_name, plot = palaeo_coordinate)
}

Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 i

In [19]:
data_epochs <- data.frame(Epoch = c("Pliocene_Holocene", "Miocene", "Oligocene", "Eocene", "Palaeocene", "Campanian_Maastrichian", "Cenomanian_Santonian", "Aptian_Albian", "Barremian_Berriasian", "Late_Jurassic", "Middle_Jurassic", "Early_Jurassic"),
        Age_min = c(0, 5.334, 23.031, 33.901, 56.001, 66.001, 83.601, 100.501, 125.001, 145.001, 163.501, 174.101), 
        Age_max = c(5.333, 23.03, 33.9, 56, 66, 83.6, 100.5, 125, 145, 163.5, 174.1, 201.3), 
        Age = c(0, 20, 30, 50, 65, 80, 95, 125, 140, 155, 165, 185))

In [20]:
for(i in 1:nrow(data_epochs)){
    temp_CM <- paste("PaleoDEMS/CM/", data_epochs$Age[i], "Ma_CM_v7.shp", sep = "")
    temp_name <- paste("Plot/PALEOMAP_DEMS_paleocoordonates/paleocoordinates_", data_epochs$Epoch[i], ".pdf", sep ="")
    temp_CS <- paste("PaleoDEMS/CS/", data_epochs$Age[i], "Ma_CS_v7.shp", sep = "")
    
    temp_paleo_data <- paleo_coordinate_cleaned[paleo_coordinate_cleaned$age > data_epochs$Age_min[i] & paleo_coordinate_cleaned$age < data_epochs$Age_max[i], ]
    pal_cor <- data.frame(longitude = temp_paleo_data$p_lng_PALEOMAP, latitude = temp_paleo_data$p_lat_PALEOMAP)
    
    gpm <- read_sf(temp_CM)

    gpm2 <- read_sf(temp_CS)
    
    coordinates(pal_cor) <- ~longitude + latitude
    proj4string(pal_cor) <- CRS("+proj=longlat +datum=WGS84")
    SpatialPoints(pal_cor, proj4string = CRS("ESRI:54030"))
    coords_robinson <- spTransform(pal_cor, CRS("+proj=robin +datum=WGS84"))
    coords_sf <- st_as_sf(coords_robinson)
    
    palaeo_coordinate <- gpm  %>% ggplot() +         
        geom_sf(data = bb, fill = "#ADD8E6", colour = NA) +
        geom_sf(data = gpm, fill = "#81CDC6", colour = "black", alpha = 1) +
        geom_sf(data = gpm2, fill = "#BEB4B0", colour = "black", alpha = 1) +
        geom_sf(data = bbox) +
        geom_sf(data = coords_sf, aes(color = "#FAC898")) +
        coord_sf(crs = sf::st_crs("ESRI:54030")) +
        theme_void()
    
    ggsave(filename = temp_name, plot = palaeo_coordinate)
}

Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image


In [21]:
data_epochs <- data.frame(Epoch = c(0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 60, 65, 70, 80, 85, 90, 95, 105, 120, 125, 130, 135, 140, 150, 155, 160, 165, 170, 180, 185, 195, 200, 205, 220, 230, 240, 245, 250),
        Age_min = c(0, 5.001, 10.001, 15.001, 20.001, 25.001, 30.001, 35.001, 40.001, 45.001, 50.001, 60.001, 65.001, 70.001, 80.001, 85.001, 90.001, 95.001, 105.001, 120.001, 125.001, 130.001, 135.001, 140.001, 150.001, 155.001, 160.001, 165.001, 170.001, 180.001, 185.001, 195.001, 200.001, 205.001, 220.001, 230.001, 240.001, 245.001, 250.001), 
        Age_max = c(5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 60, 65, 70, 80, 85, 90, 95, 105, 120, 125, 130, 135, 140, 150, 155, 160, 165, 170, 180, 185, 195, 200, 205, 220, 230, 240, 245, 250, 255), 
        Age = c(0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 60, 65, 70, 80, 85, 90, 95, 105, 120, 125, 130, 135, 140, 150, 155, 160, 165, 170, 180, 185, 195, 200, 205, 220, 230, 240, 245, 250))

In [22]:
for(i in 1:nrow(data_epochs)){
    temp_CM <- paste("PaleoDEMS/CM/", data_epochs$Age[i], "Ma_CM_v7.shp", sep = "")
    temp_name <- paste("Plot/PALEOMAP_DEMS_paleocoordonates/paleocoordinates_", data_epochs$Epoch[i], ".pdf", sep ="")
    temp_CS <- paste("PaleoDEMS/CS/", data_epochs$Age[i], "Ma_CS_v7.shp", sep = "")
    
    temp_paleo_data <- paleo_coordinate_cleaned[paleo_coordinate_cleaned$age > data_epochs$Age_min[i] & paleo_coordinate_cleaned$age < data_epochs$Age_max[i], ]
    pal_cor <- data.frame(longitude = temp_paleo_data$p_lng_PALEOMAP, latitude = temp_paleo_data$p_lat_PALEOMAP)
    
    gpm <- read_sf(temp_CM)

    gpm2 <- read_sf(temp_CS)
    
    
    palaeo_coordinate <- gpm  %>% ggplot() +         
        geom_sf(data = bb, fill = "#ADD8E6", colour = NA) +
        geom_sf(data = gpm, fill = "#81CDC6", colour = "black", alpha = 1) +
        geom_sf(data = gpm2, fill = "#BEB4B0", colour = "black", alpha = 1) +
        geom_sf(data = bbox) +
        coord_sf(crs = sf::st_crs("ESRI:54030")) +
        theme_void()    
    


    if(nrow(temp_paleo_data) != 0){
        
            coordinates(pal_cor) <- ~longitude + latitude
            proj4string(pal_cor) <- CRS("+proj=longlat +datum=WGS84")
            SpatialPoints(pal_cor, proj4string = CRS("ESRI:54030"))
            coords_robinson <- spTransform(pal_cor, CRS("+proj=robin +datum=WGS84"))
            coords_sf <- st_as_sf(coords_robinson)
    
            palaeo_coordinate <- gpm  %>% ggplot() +         
                geom_sf(data = bb, fill = "#ADD8E6", colour = NA) +
                geom_sf(data = gpm, fill = "#81CDC6", colour = "black", alpha = 1) +
                geom_sf(data = gpm2, fill = "#BEB4B0", colour = "black", alpha = 1) +
                geom_sf(data = coords_sf, aes(color = "#FAC898")) + 
                geom_sf(data = bbox) +
                coord_sf(crs = sf::st_crs("ESRI:54030")) +
                theme_void()
      
        
    }
    
    ggsave(filename = temp_name, plot = palaeo_coordinate)
}

Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 i

## Extant occurences

In [23]:
table_coordinate_extant_without_whale_shark <- table_coordinate_extant[!table_coordinate_extant$Species == "Rhincodon typus", ]

In [24]:
table_coordinate_extant_without_whale_shark <- table_coordinate_extant[table_coordinate_extant_without_whale_shark$References == "Inaturalist", ]

In [25]:
table_coordinate_extant_subsampled <- table_coordinate_extant_without_whale_shark %>% 
      group_by(Species) %>%
          slice(sample(min(10, n())))

In [26]:
unique_sp <- unique(table_coordinate_extant_without_whale_shark$Species)

for(i in 1:length(unique_sp)){
    
    temp <- table_coordinate_extant_without_whale_shark[table_coordinate_extant_without_whale_shark$Species == unique_sp[i], ]

    
    if(i == 1){
        if(nrow(temp) < 5){
            table_coordinate_extant_subsampled <- temp[,c(4,5)]
        }
        
        else{
            table_coordinate_extant_subsampled <- clustr(dat = temp, xy = c("Longitude", "Latitude"), iter = 1, nSite = 5, distMax = 1000)[[1]]
        }

    }
    
    else{
        
        if(nrow(temp) < 5){
            table_coordinate_extant_subsampled <- rbind(table_coordinate_extant_subsampled,temp[,c(4,5)])
        }
        else{
            table_coordinate_extant_subsampled <- rbind(table_coordinate_extant_subsampled, clustr(dat = temp, xy = c("Longitude", "Latitude"), iter = 1, nSite = 5, distMax = 1000)[[1]])
        }
    }
    
}

Warning message in min(mins):
“aucun argument trouvé pour min ; Inf est renvoyé”
Warning message in min(mins):
“aucun argument trouvé pour min ; Inf est renvoyé”
Warning message in min(mins):
“aucun argument trouvé pour min ; Inf est renvoyé”
Warning message in min(mins):
“aucun argument trouvé pour min ; Inf est renvoyé”
Warning message in min(mins):
“aucun argument trouvé pour min ; Inf est renvoyé”
Warning message in min(mins):
“aucun argument trouvé pour min ; Inf est renvoyé”
Warning message in min(mins):
“aucun argument trouvé pour min ; Inf est renvoyé”
Warning message in min(mins):
“aucun argument trouvé pour min ; Inf est renvoyé”
Warning message in min(mins):
“aucun argument trouvé pour min ; Inf est renvoyé”
Warning message in min(mins):
“aucun argument trouvé pour min ; Inf est renvoyé”
Warning message in min(mins):
“aucun argument trouvé pour min ; Inf est renvoyé”
Warning message in min(mins):
“aucun argument trouvé pour min ; Inf est renvoyé”
Warning message in min(mins)

In [27]:
for(i in 1:nrow(data_epochs)){
    temp_CM <- paste("PaleoDEMS/CM/", data_epochs$Age[i], "Ma_CM_v7.shp", sep = "")
    temp_name <- paste("Plot/PALEOMAP_DEMS_paleocoordonates/paleocoordinates_", data_epochs$Epoch[i], ".pdf", sep ="")
    temp_CS <- paste("PaleoDEMS/CS/", data_epochs$Age[i], "Ma_CS_v7.shp", sep = "")
    
    temp_paleo_data <- paleo_coordinate_cleaned[paleo_coordinate_cleaned$age > data_epochs$Age_min[i] & paleo_coordinate_cleaned$age < data_epochs$Age_max[i], ]
    pal_cor <- data.frame(longitude = temp_paleo_data$p_lng_PALEOMAP, latitude = temp_paleo_data$p_lat_PALEOMAP)
    
    gpm <- read_sf(temp_CM)

    gpm2 <- read_sf(temp_CS)
        
        coordinates(pal_cor) <- ~longitude + latitude
        proj4string(pal_cor) <- CRS("+proj=longlat +datum=WGS84")
        SpatialPoints(pal_cor, proj4string = CRS("ESRI:54030"))
        coords_robinson <- spTransform(pal_cor, CRS("+proj=robin +datum=WGS84"))
        coords_sf <- st_as_sf(coords_robinson)
        
        coordinates(cor_extant) <- ~longitude + latitude
        proj4string(cor_extant) <- CRS("+proj=longlat +datum=WGS84")
        SpatialPoints(cor_extant, proj4string = CRS("ESRI:54030"))
        coords_robinson_extant <- spTransform(cor_extant, CRS("+proj=robin +datum=WGS84"))
        cor_extant_sf <- st_as_sf(coords_robinson_extant)
        
    
            palaeo_coordinate <- gpm  %>% ggplot() +         
                geom_sf(data = bb, fill = "#ADD8E6", colour = NA) +
                geom_sf(data = gpm, fill = "#81CDC6", colour = "black", alpha = 1) +
                geom_sf(data = gpm2, fill = "#BEB4B0", colour = "black", alpha = 1) +
                geom_sf(data = coords_sf, aes(color = "#FAC898")) +
                geom_sf(data = cor_extant_sf, colour = "#CB99CC") +
                geom_sf(data = bbox) +
                coord_sf(crs = sf::st_crs("ESRI:54030")) +
                theme_void()
      
        
    }
    
    ggsave(filename = temp_name, plot = palaeo_coordinate)
}

ERROR: Error in parse(text = x, srcfile = src): <text>:40:1: '}' inattendu(e)
39:     ggsave(filename = temp_name, plot = palaeo_coordinate)
40: }
    ^


In [ ]:
pdf("Paleo_lat.pdf")
plot(x=paleo_coordinate_cleaned$age, y=paleo_coordinate_cleaned$p_lat_PALEOMAP, axes = FALSE, xlab = NA, ylab = "Latitude (°)", col = "#FAC898", pch = 16)
points(x=rep(0, nrow(table_coordinate_extant_subsampled)), y=table_coordinate_extant_subsampled$Latitude,  col = "#CB99CC", pch = 16)
lines(lowess(x=c(paleo_coordinate_cleaned$age,rep(0, nrow(table_coordinate_extant_subsampled))), y=c(paleo_coordinate_cleaned$p_lat_PALEOMAP, table_coordinate_extant_subsampled$Latitude)), col = 3, lwd = 3)
box()
axis(2)
axis_geo(side = 1, intervals = list("epoch", "period"),
    abbr = list(TRUE, FALSE))
dev.off()

## Save dataframes

In [ ]:
write.table(table_coordinate_extant_subsampled, "subsampled_extant_diversity.tsv", sep ="\t", row.names = FALSE, quote = FALSE)

In [ ]:
write.table(paleo_coordinate_cleaned, "paleo_occurences", sep ="\t", row.names = FALSE, quote = FALSE)